In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori,association_rules

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DataSet/lecture_convenience_store_10.csv")
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   bill_id  31 non-null     int64 
 1   item     31 non-null     object
dtypes: int64(1), object(1)
memory usage: 628.0+ bytes


,bill_id,item
0,1,Bread
1,1,Milk
2,1,Eggs
3,2,Bread
4,2,Butter


In [ ]:
#แปลงข้อมูลให้อยู่ในรูปแบบของ Basket
basket = (df.groupby(['bill_id','item'])['item']
          .count()
          .unstack()
          .reset_index()
          .fillna(0)
          .set_index('bill_id')
          )
basket

item,Bread,Butter,Coffee,Eggs,Milk,Sugar
bill_id,,,,,,
1,1.0,0.0,0.0,1.0,1.0,0.0
2,1.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,1.0,1.0
4,1.0,1.0,0.0,0.0,1.0,0.0
5,1.0,1.0,0.0,1.0,1.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0
7,1.0,0.0,1.0,0.0,1.0,1.0
8,1.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,1.0,1.0,1.0,1.0


In [ ]:
#แปลงจากตัวเลขให้เป็น Boolean
basket_encoded = basket.map(lambda x: x>0)
basket_encoded

item,Bread,Butter,Coffee,Eggs,Milk,Sugar
bill_id,,,,,,
1,True,False,False,True,True,False
2,True,True,False,False,False,False
3,False,False,True,False,True,True
4,True,True,False,False,True,False
5,True,True,False,True,True,False
6,False,False,True,False,False,True
7,True,False,True,False,True,True
8,True,False,False,True,False,False
9,False,False,True,True,True,True


In [ ]:
#หาชุดสินค้าที่ซื้อร่วมกันบ่อยๆ
frequent_itemset = apriori(basket_encoded,min_support=0.3,use_colnames=True)
#สร้างกฏความสัมพันธ์
rules = association_rules(frequent_itemset,metric='lift',min_threshold=1)
rules.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Butter),(Bread),0.4,0.7,0.4,1.000000,1.428571,1.0,0.12,inf,0.500000,0.571429,1.000000,0.785714
1,(Bread),(Butter),0.7,0.4,0.4,0.571429,1.428571,1.0,0.12,1.400000,1.000000,0.571429,0.285714,0.785714
2,(Bread),(Eggs),0.7,0.5,0.4,0.571429,1.142857,1.0,0.05,1.166667,0.416667,0.500000,0.142857,0.685714
3,(Eggs),(Bread),0.5,0.7,0.4,0.800000,1.142857,1.0,0.05,1.500000,0.250000,0.500000,0.333333,0.685714
4,(Bread),(Milk),0.7,0.7,0.5,0.714286,1.020408,1.0,0.01,1.050000,0.066667,0.555556,0.047619,0.714286
5,(Milk),(Bread),0.7,0.7,0.5,0.714286,1.020408,1.0,0.01,1.050000,0.066667,0.555556,0.047619,0.714286
6,(Butter),(Milk),0.4,0.7,0.3,0.750000,1.071429,1.0,0.02,1.200000,0.111111,0.375000,0.166667,0.589286
7,(Milk),(Butter),0.7,0.4,0.3,0.428571,1.071429,1.0,0.02,1.050000,0.222222,0.375000,0.047619,0.589286
8,(Milk),(Coffee),0.7,0.4,0.3,0.428571,1.071429,1.0,0.02,1.050000,0.222222,0.375000,0.047619,0.589286
9,(Coffee),(Milk),0.4,0.7,0.3,0.750000,1.071429,1.0,0.02,1.200000,0.111111,0.375000,0.166667,0.589286


In [ ]:
#คัดเฉพาะกฏที่สำคัญ
rules_sort = rules[['antecedents','consequents','support','confidence','lift']].sort_values(
    by=['lift','confidence'],ascending = False
)
print("กฏความสัมพันธ์ที่ค้นพบ")
print(rules_sort.to_string())

กฏความสัมพันธ์ที่ค้นพบ
        antecedents      consequents  support  confidence      lift
10          (Sugar)         (Coffee)      0.4    1.000000  2.500000
11         (Coffee)          (Sugar)      0.4    1.000000  2.500000
28    (Sugar, Milk)         (Coffee)      0.3    1.000000  2.500000
29   (Coffee, Milk)          (Sugar)      0.3    1.000000  2.500000
32          (Sugar)   (Coffee, Milk)      0.3    0.750000  2.500000
33         (Coffee)    (Sugar, Milk)      0.3    0.750000  2.500000
19         (Butter)    (Bread, Milk)      0.3    0.750000  1.500000
18    (Bread, Milk)         (Butter)      0.3    0.600000  1.500000
20          (Bread)   (Butter, Milk)      0.3    0.428571  1.428571
0          (Butter)          (Bread)      0.4    1.000000  1.428571
17   (Butter, Milk)          (Bread)      0.3    1.000000  1.428571
1           (Bread)         (Butter)      0.4    0.571429  1.428571
22    (Bread, Milk)           (Eggs)      0.3    0.600000  1.200000
27           (Eggs)    (B